# Обучение детектора промышленных деталей
**Перед запуском:** загрузи `dataset.zip` на Google Drive.
```
# Локально (в корне проекта):
Compress-Archive -Path dataset -DestinationPath dataset.zip
```

In [ ]:
# ── 1. GPU и Drive ─────────────────────────────────────────────────────────
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ — смени Runtime → GPU')
print('CUDA:', torch.version.cuda)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Пути — НАСТРОЙ ПОД СЕБЯ ─────────────────────────────────────────────
# Путь к dataset.zip на Google Drive
DRIVE_DATASET_ZIP = '/content/drive/MyDrive/dissertation/dataset.zip'

# Куда сохранять веса (на Drive, чтобы не потерять при перезапуске)
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/dissertation/weights'

# Гиперпараметры обучения
EPOCHS     = 50
BATCH_SIZE = 32   # T4: 32 комфортно; если OOM — уменьши до 16
LR         = 1e-3
NUM_CLASSES = 3

import os
os.makedirs(DRIVE_WEIGHTS_DIR, exist_ok=True)
print('Настройки OK')

In [ ]:
# ── 3. Распаковка датасета ──────────────────────────────────────────────────
import zipfile, os

DATASET_ROOT = '/content/dataset'

if not os.path.exists(DATASET_ROOT):
    print('Распаковываю датасет...')
    with zipfile.ZipFile(DRIVE_DATASET_ZIP, 'r') as z:
        z.extractall('/content/')
    print('Готово')
else:
    print('Датасет уже распакован')

# Проверка структуры
for split in ['train', 'val', 'test']:
    n = len(list(__import__('pathlib').Path(f'{DATASET_ROOT}/images/{split}').glob('*.png')))
    print(f'  {split}: {n} изображений')

In [ ]:
%%writefile /content/architecture.py
"""
Архитектура кастомного CNN-детектора промышленных деталей.

Одностадийный детектор на основе сетки 20x20.
Каждая ячейка предсказывает:
  - objectness: вероятность наличия объекта (1 значение)
  - bbox:        tx, ty — смещение центра внутри ячейки; tw, th — размеры
  - class:       логиты классов (C значений)

Декодирование bbox из ячейки (gi=строка, gj=столбец), G=20:
  x_c = (gj + sigmoid(tx)) / G
  y_c = (gi + sigmoid(ty)) / G
  w   = sigmoid(tw)
  h   = sigmoid(th)
"""

import torch
import torch.nn as nn


class ConvBNReLU(nn.Module):
    """Базовый строительный блок: Conv2d → BatchNorm2d → ReLU."""

    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock(nn.Module):
    """
    Остаточный блок: два слоя ConvBNReLU + skip connection.
    Input/Output: (B, C, H, W) -> (B, C, H, W)
    """

    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNReLU(channels, channels, 3, 1, 1),
            ConvBNReLU(channels, channels, 3, 1, 1),
        )

    def forward(self, x):
        return x + self.block(x)


class Backbone(nn.Module):
    """
    Backbone: последовательное уменьшение разрешения stride=2 свёртками.

    Input:  (B,   3, 640, 640)
    stage1: (B,  32, 320, 320)  <- stride=2
    stage2: (B,  64, 160, 160)  <- stride=2 + ResBlock x1
    stage3: (B, 128,  80,  80)  <- stride=2 + ResBlock x2
    stage4: (B, 256,  40,  40)  <- stride=2 + ResBlock x2
    stage5: (B, 512,  20,  20)  <- stride=2 + ResBlock x1
    Output: (B, 512,  20,  20)
    """

    def __init__(self):
        super().__init__()
        self.stage1 = nn.Sequential(
            ConvBNReLU(3,  32, kernel_size=3, stride=2, padding=1),
            ConvBNReLU(32, 32, kernel_size=3, stride=1, padding=1),
        )
        self.stage2 = nn.Sequential(
            ConvBNReLU(32, 64, kernel_size=3, stride=2, padding=1),
            ResidualBlock(64),
        )
        self.stage3 = nn.Sequential(
            ConvBNReLU(64, 128, kernel_size=3, stride=2, padding=1),
            ResidualBlock(128),
            ResidualBlock(128),
        )
        self.stage4 = nn.Sequential(
            ConvBNReLU(128, 256, kernel_size=3, stride=2, padding=1),
            ResidualBlock(256),
            ResidualBlock(256),
        )
        self.stage5 = nn.Sequential(
            ConvBNReLU(256, 512, kernel_size=3, stride=2, padding=1),
            ResidualBlock(512),
        )

    def forward(self, x):
        x = self.stage1(x)  # (B,  32, 320, 320)
        x = self.stage2(x)  # (B,  64, 160, 160)
        x = self.stage3(x)  # (B, 128,  80,  80)
        x = self.stage4(x)  # (B, 256,  40,  40)
        x = self.stage5(x)  # (B, 512,  20,  20)
        return x


class DetectionHead(nn.Module):
    """
    Голова детекции: формирует предсказания для каждой ячейки сетки.

    Выходной вектор на ячейку: [obj, tx, ty, tw, th, cls0, cls1, cls2]
    Input:  (B, 512, 20, 20)
    Output: (B, 1+4+C, 20, 20)
    """

    def __init__(self, in_ch=512, num_classes=3):
        super().__init__()
        out_ch = 1 + 4 + num_classes
        self.head = nn.Sequential(
            ConvBNReLU(in_ch, 256, kernel_size=3, stride=1, padding=1),
            ConvBNReLU(256,   128, kernel_size=1, stride=1, padding=0),
            nn.Conv2d(128, out_ch, kernel_size=1, stride=1, padding=0),
        )

    def forward(self, x):
        return self.head(x)


class Detector(nn.Module):
    """
    Полная модель: Backbone + DetectionHead.
    Input:  (B, 3, 640, 640)
    Output: (B, 1+4+C, 20, 20)
    """

    GRID_SIZE = 20

    def __init__(self, num_classes=3):
        super().__init__()
        self.num_classes = num_classes
        self.backbone = Backbone()
        self.head = DetectionHead(in_ch=512, num_classes=num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

    def predict(self, x, conf_thresh=0.5):
        self.eval()
        with torch.no_grad():
            raw = self.forward(x)
        G = self.GRID_SIZE
        results = []
        for b in range(x.shape[0]):
            pred = raw[b]
            obj_map = torch.sigmoid(pred[0])
            max_conf = obj_map.max().item()
            if max_conf < conf_thresh:
                results.append(None)
                continue
            flat_idx = obj_map.argmax()
            gi = (flat_idx // G).item()
            gj = (flat_idx %  G).item()
            tx = torch.sigmoid(pred[1, gi, gj]).item()
            ty = torch.sigmoid(pred[2, gi, gj]).item()
            tw = torch.sigmoid(pred[3, gi, gj]).item()
            th = torch.sigmoid(pred[4, gi, gj]).item()
            x_c = (gj + tx) / G
            y_c = (gi + ty) / G
            cls = int(pred[5:, gi, gj].argmax().item())
            results.append({'class': cls, 'conf': float(max_conf), 'bbox': [x_c, y_c, tw, th]})
        return results

In [ ]:
%%writefile /content/loss.py
"""
Функция потерь детектора.

Компоненты:
  1. Objectness loss — BCE с разными весами для pos/neg ячеек
  2. Bbox loss       — Smooth L1, только pos ячейки
  3. Class loss      — CrossEntropy, только pos ячейки
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class DetectionLoss(nn.Module):

    def __init__(self, grid_size=20, num_classes=3,
                 lambda_obj=1.0, lambda_noobj=0.5,
                 lambda_bbox=5.0, lambda_cls=1.0):
        super().__init__()
        self.G            = grid_size
        self.num_classes  = num_classes
        self.lambda_obj   = lambda_obj
        self.lambda_noobj = lambda_noobj
        self.lambda_bbox  = lambda_bbox
        self.lambda_cls   = lambda_cls

    def forward(self, predictions, targets):
        B, _, G, _ = predictions.shape
        device = predictions.device

        obj_mask = torch.zeros(B, G, G, device=device)
        tx_tgt   = torch.zeros(B, G, G, device=device)
        ty_tgt   = torch.zeros(B, G, G, device=device)
        tw_tgt   = torch.zeros(B, G, G, device=device)
        th_tgt   = torch.zeros(B, G, G, device=device)
        cls_tgt  = torch.zeros(B, G, G, dtype=torch.long, device=device)

        for b, tgt in enumerate(targets):
            if tgt is None:
                continue
            x_c, y_c, w, h = tgt['bbox']
            cls = int(tgt['class'])
            gj = min(int(x_c * G), G - 1)
            gi = min(int(y_c * G), G - 1)
            obj_mask[b, gi, gj] = 1.0
            tx_tgt[b, gi, gj]   = x_c * G - gj
            ty_tgt[b, gi, gj]   = y_c * G - gi
            tw_tgt[b, gi, gj]   = w
            th_tgt[b, gi, gj]   = h
            cls_tgt[b, gi, gj]  = cls

        noobj_mask = 1.0 - obj_mask

        # Objectness loss
        pred_obj = torch.sigmoid(predictions[:, 0, :, :])
        bce = F.binary_cross_entropy(pred_obj, obj_mask, reduction='none')
        obj_loss = (
            self.lambda_obj   * (bce * obj_mask).sum() +
            self.lambda_noobj * (bce * noobj_mask).sum()
        ) / B

        # Bbox loss
        pred_bbox = torch.sigmoid(predictions[:, 1:5, :, :])
        bbox_tgt  = torch.stack([tx_tgt, ty_tgt, tw_tgt, th_tgt], dim=1)
        bbox_loss_map = F.smooth_l1_loss(pred_bbox, bbox_tgt, reduction='none').sum(dim=1)
        n_obj = obj_mask.sum().clamp(min=1.0)
        bbox_loss = self.lambda_bbox * (bbox_loss_map * obj_mask).sum() / n_obj

        # Class loss
        pred_cls    = predictions[:, 5:, :, :]
        pos_indices = obj_mask.bool()
        if pos_indices.any():
            pred_pos = pred_cls.permute(0, 2, 3, 1)[pos_indices]
            cls_pos  = cls_tgt[pos_indices]
            cls_loss = self.lambda_cls * F.cross_entropy(pred_pos, cls_pos)
        else:
            cls_loss = torch.tensor(0.0, device=device)

        total = obj_loss + bbox_loss + cls_loss
        return total, {'obj': obj_loss.item(), 'bbox': bbox_loss.item(), 'cls': cls_loss.item()}

In [ ]:
# ── 6. Dataset и обучение ───────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content')

import csv
import shutil
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from architecture import Detector
from loss import DetectionLoss


class PartDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, img_size=640):
        self.img_size  = img_size
        self.lbl_dir   = Path(lbl_dir)
        self.img_paths = sorted(
            p for p in Path(img_dir).glob('*.png')
            if (self.lbl_dir / (p.stem + '.txt')).exists()
        )

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        lbl_path = self.lbl_dir / (img_path.stem + '.txt')
        img_bgr = cv2.imread(str(img_path))
        img_bgr = cv2.resize(img_bgr, (self.img_size, self.img_size))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_t   = torch.from_numpy(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1)
        line    = open(lbl_path).readline().strip().split()
        cls     = int(line[0])
        x_c, y_c, w, h = float(line[1]), float(line[2]), float(line[3]), float(line[4])
        return img_t, {'class': cls, 'bbox': [x_c, y_c, w, h]}


def collate_fn(batch):
    imgs, targets = zip(*batch)
    return torch.stack(imgs), list(targets)


print('Dataset и модули загружены')

In [ ]:
# ── 7. Запуск обучения ──────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')

dataset_root = Path(DATASET_ROOT)
save_dir     = Path(DRIVE_WEIGHTS_DIR)

train_ds = PartDataset(
    img_dir=str(dataset_root / 'images' / 'train'),
    lbl_dir=str(dataset_root / 'labels' / 'train'),
)
val_ds = PartDataset(
    img_dir=str(dataset_root / 'images' / 'val'),
    lbl_dir=str(dataset_root / 'labels' / 'val'),
)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn, pin_memory=True)

model     = Detector(num_classes=NUM_CLASSES).to(device)
criterion = DetectionLoss(num_classes=NUM_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

log_path = save_dir / 'train_log.csv'
with open(log_path, 'w', newline='') as f:
    csv.writer(f).writerow(['epoch', 'train_loss', 'val_loss', 'lr'])

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss_sum = 0.0
    for batch_idx, (imgs, targets) in enumerate(train_loader):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss, components = criterion(preds, targets)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        train_loss_sum += loss.item()

    # Val
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device)
            preds = model(imgs)
            loss, _ = criterion(preds, targets)
            val_loss_sum += loss.item()

    train_avg = train_loss_sum / len(train_loader)
    val_avg   = val_loss_sum   / len(val_loader)
    cur_lr    = optimizer.param_groups[0]['lr']

    print(f'[{epoch:02d}/{EPOCHS}] train={train_avg:.4f}  val={val_avg:.4f}  lr={cur_lr:.1e}')

    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, train_avg, val_avg, cur_lr])

    if val_avg < best_val_loss:
        best_val_loss = val_avg
        ckpt_path = save_dir / 'detector_best.pt'
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': val_avg}, ckpt_path)
        print(f'  → Checkpoint сохранён: {ckpt_path}')

    scheduler.step(val_avg)

print(f'\nОбучение завершено. Лучший val_loss={best_val_loss:.4f}')

In [ ]:
# ── 8. График train/val loss ────────────────────────────────────────────────
import csv
import matplotlib.pyplot as plt

epochs_log, train_losses, val_losses = [], [], []
with open(save_dir / 'train_log.csv') as f:
    for row in csv.DictReader(f):
        epochs_log.append(int(row['epoch']))
        train_losses.append(float(row['train_loss']))
        val_losses.append(float(row['val_loss']))

plt.figure(figsize=(10, 4))
plt.plot(epochs_log, train_losses, label='Train loss')
plt.plot(epochs_log, val_losses,   label='Val loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Кривые обучения детектора')
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig(str(save_dir / 'loss_curve.png'), dpi=150)
plt.show()
print('График сохранён в Drive')

In [ ]:
# ── 9. Быстрая оценка на test-сплите ───────────────────────────────────────
CLASS_NAMES = {0: 'gaika', 1: 'vilka', 2: 'vtulka'}

# Загружаем лучший checkpoint
ckpt = torch.load(save_dir / 'detector_best.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Загружены веса эпохи {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

test_ds = PartDataset(
    img_dir=str(dataset_root / 'images' / 'test'),
    lbl_dir=str(dataset_root / 'labels' / 'test'),
)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False,
                         num_workers=2, collate_fn=collate_fn)
print(f'Test: {len(test_ds)} изображений')

# Простая accuracy по классам
correct = 0
total   = 0
class_correct = {i: 0 for i in range(NUM_CLASSES)}
class_total   = {i: 0 for i in range(NUM_CLASSES)}

with torch.no_grad():
    for imgs, targets in test_loader:
        imgs    = imgs.to(device)
        results = model.predict(imgs, conf_thresh=0.3)
        for result, tgt in zip(results, targets):
            gt_cls = tgt['class']
            class_total[gt_cls] += 1
            total += 1
            if result is not None and result['class'] == gt_cls:
                correct += 1
                class_correct[gt_cls] += 1

print(f'\nОбщая accuracy: {correct/total:.4f} ({correct}/{total})')
print('По классам:')
for cls_id in range(NUM_CLASSES):
    ct = class_total[cls_id]
    cc = class_correct[cls_id]
    print(f'  {CLASS_NAMES[cls_id]:8s}: {cc/ct:.4f}  ({cc}/{ct})')

In [ ]:
# ── 10. Скачать веса локально ───────────────────────────────────────────────
# Запусти эту ячейку чтобы скачать detector_best.pt напрямую из Colab
from google.colab import files
files.download(str(save_dir / 'detector_best.pt'))
files.download(str(save_dir / 'train_log.csv'))